# A. Ablation: 증강 범위 · 녹화 앞뒤 구간 · 구간 길이
모든 실험은 **녹화 하나 빼기 21-fold**, 예측을 모은 뒤 한 번에 평가한다.

- 모델: B와 같은 설정을 쓰기 위해 `b_classic.make_classic()`을 그대로 사용
  - `logreg_l2` (팀 고전 모델 최고, 특성 A+C), `rf` (같은 특성)
- 기준 결과(아무것도 안 한 상태)는 B의 results.csv 값과 같아야 한다

| 실험 | 바꾸는 것 | 질문 |
|---|---|---|
| 1. 증강 범위 | 없음 / 크기 ±10·30·50% / 64스텝 겹침 | 증강이 새 녹화 일반화에 도움이 되나 |
| 2. 녹화 앞뒤 구간 | 학습에서 앞뒤 0·1·2·5구간 제외 | 출발·정지 구간이 학습을 방해하나 |
| 3. 구간 길이 | 32 / 64 / 128 스텝 | 바닥이 바뀐 뒤 얼마나 빨리 알아챌 수 있나 |

In [ ]:
%cd /content
!test -d ML-imu || git clone -q https://github.com/ekgP525/ML-imu.git
%cd /content/ML-imu
!git pull -q
!pip install -q -r requirements.txt
!test -f data/X_data.npy || python data/download.py

In [ ]:
import sys; sys.path += ['src', 'notebooks/B_classic']
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recordings import load_dataset, load_chunks
from splits import load_folds, fold_indices
from preprocess import overlap_windows, trim_edges
from features import extract_features
from evaluate import evaluate, log_result
from b_classic import make_classic, select_features

SEED = 42
FEAT = 'AC'
MODELS = ['logreg_l2', 'rf']

X, y, rec = load_dataset('data')
chunk = load_chunks('data')
folds = load_folds()
F_AC = select_features(extract_features(X), FEAT)
print(X.shape, F_AC.shape, len(folds))

def feats(x):
    return select_features(extract_features(x), FEAT)

def pooled(model, pairs):
    """pairs: fold마다 (F_train, y_train, F_test, y_test) → 예측을 모아 평가"""
    yt, yp = [], []
    for Ftr, ytr, Fte, yte in pairs:
        yp += list(make_classic(model).fit(Ftr, ytr).predict(Fte))
        yt += list(yte)
    return evaluate(yt, yp), np.array(yt), np.array(yp)

## 0. 기준 결과 (B의 results.csv와 같은지 확인)

In [ ]:
base = {}
for m in MODELS:
    pairs = [(F_AC.iloc[tr], y[tr], F_AC.iloc[te], y[te]) for tr, te in (fold_indices(f, rec) for f in folds)]
    r, yt, yp = pooled(m, pairs)
    base[m] = (r, yt, yp)
    print(m, {k: round(v, 4) for k, v in r.items()})
# B 기록: logreg_l2 8class_AC macro-F1 0.4039, rf 8class_AC 0.3082

## 1. 증강 범위
- 크기 조절은 **진동 성분만** 키우고 줄인다: `(x − 구간 평균) × s + 구간 평균`.
  원본 신호를 통째로 곱하면 중력(약 9.8)까지 바뀌어 실제로는 없는 기울기 변화가 생기기 때문
- 잡음은 채널별 진동 표준편차의 5%, 시간 이동은 ±8스텝
- 겹침: 같은 묶음(`load_chunks`)의 구간을 이어 붙여 64스텝 간격으로 다시 자름
- 증강은 **학습 녹화에만** 적용하고, 테스트는 원본 그대로

In [ ]:
def vib_augment(x, scale, noise_rel=0.05, max_shift=8, seed=SEED):
    rng = np.random.default_rng(seed)
    mu = x.mean(1, keepdims=True)
    a = (x - mu) * rng.uniform(1 - scale, 1 + scale, size=(len(x), 1, 1)) + mu
    shifts = rng.integers(-max_shift, max_shift + 1, size=len(x))
    a = np.stack([np.roll(v, k, axis=0) for v, k in zip(a, shifts)])
    a = a + rng.normal(size=a.shape) * (x - mu).std(axis=(0, 1)) * noise_rel
    return a.astype(np.float32)

def aug_pairs(cond):
    for f in folds:
        tr, te = fold_indices(f, rec)
        if cond == 'overlap':
            xa, ya, _ = overlap_windows(X[tr], y[tr], chunk[tr])
        else:
            xa = np.concatenate([X[tr], vib_augment(X[tr], cond)])
            ya = np.concatenate([y[tr], y[tr]])
        yield feats(xa), ya, F_AC.iloc[te], y[te]

aug_rows = [{'model': m, 'aug': 'none', **base[m][0]} for m in MODELS]
for cond in [0.1, 0.3, 0.5, 'overlap']:
    for m in MODELS:
        r, _, _ = pooled(m, aug_pairs(cond))
        label = 'overlap64' if cond == 'overlap' else f'scale±{int(cond * 100)}%'
        aug_rows.append({'model': m, 'aug': label, **r})
        log_result('A', m, r, setting=f'ablation_aug={label}|feat={FEAT}', note='증강 ablation')
        print(m, label, round(r['macro_f1'], 4))
aug_res = pd.DataFrame(aug_rows)
display(aug_res.pivot(index='aug', columns='model', values='macro_f1').loc[
    ['none', 'scale±10%', 'scale±30%', 'scale±50%', 'overlap64']].round(4))

## 2. 녹화 앞뒤 구간
- 각 녹화의 처음·마지막 n구간은 카트가 출발·정지하는 구간일 가능성이 있다
- **학습에서만** 빼고, 테스트는 전체 구간 그대로 평가 (평가 대상을 같게 유지)
- 추가로 기준 모델이 앞뒤 구간과 가운데 구간에서 정확도가 다른지도 확인

In [ ]:
def edge_pairs(n):
    for f in folds:
        tr, te = fold_indices(f, rec)
        _, _, _, keep = trim_edges(X[tr], y[tr], rec[tr], n=n)
        yield F_AC.iloc[tr].iloc[keep], y[tr][keep], F_AC.iloc[te], y[te]

edge_rows = [{'model': m, 'trim': 0, **base[m][0]} for m in MODELS]
for n in [1, 2, 5]:
    for m in MODELS:
        r, _, _ = pooled(m, edge_pairs(n))
        edge_rows.append({'model': m, 'trim': n, **r})
        log_result('A', m, r, setting=f'ablation_trim={n}|feat={FEAT}', note='녹화 앞뒤 구간 학습 제외')
        print(m, f'trim={n}', round(r['macro_f1'], 4))
edge_res = pd.DataFrame(edge_rows)
display(edge_res.pivot(index='trim', columns='model', values='macro_f1').round(4))

# 기준 모델: 앞뒤 2구간 vs 가운데 구간 정확도 (fold 예측은 데이터셋 순서로 정렬해 비교)
order = np.concatenate([fold_indices(f, rec)[1] for f in folds])
_, _, _, mid = trim_edges(X, y, rec, n=2)
is_edge = ~np.isin(np.arange(len(y)), mid)
for m in MODELS:
    _, yt, yp = base[m]
    correct = np.empty(len(y), dtype=bool); correct[order] = (yt == yp)
    print(f'{m}: 앞뒤 2구간 정확도 {correct[is_edge].mean():.3f} (n={is_edge.sum()}), 가운데 {correct[~is_edge].mean():.3f}')

## 3. 구간 길이
- 128스텝 구간을 겹치지 않게 L스텝으로 잘라 특성을 다시 계산한다 (L = 32, 64, 128)
- 학습·테스트 모두 같은 길이로 맞추고, 녹화 번호는 원래 구간의 것을 그대로 쓴다
- 짧을수록 바닥이 바뀐 직후에 빨리 판단할 수 있지만 정보는 줄어든다

In [ ]:
len_rows = []
for L in [32, 64, 128]:
    k = 128 // L
    XL = X.reshape(len(X), k, L, X.shape[-1]).reshape(-1, L, X.shape[-1])
    yL, recL = np.repeat(y, k), np.repeat(rec, k)
    FL = feats(XL)
    for m in MODELS:
        pairs = [(FL.iloc[tr], yL[tr], FL.iloc[te], yL[te]) for tr, te in (fold_indices(f, recL) for f in folds)]
        r, _, _ = pooled(m, pairs)
        len_rows.append({'model': m, 'length': L, **r})
        log_result('A', m, r, setting=f'ablation_len={L}|feat={FEAT}', note='구간 길이 ablation')
        print(m, f'L={L}', round(r['macro_f1'], 4))
len_res = pd.DataFrame(len_rows)
display(len_res.pivot(index='length', columns='model', values='macro_f1').round(4))

## 4. 그래프

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
aug_order = ['none', 'scale±10%', 'scale±30%', 'scale±50%', 'overlap64']
aug_plot = aug_res.pivot(index='aug', columns='model', values='macro_f1').loc[aug_order]
aug_plot.index = ['none', 'scale 10%', 'scale 30%', 'scale 50%', 'overlap 64']
aug_plot.plot(kind='bar', ax=axes[0], rot=0, title='1. Augmentation')
edge_res.pivot(index='trim', columns='model', values='macro_f1').plot(
    kind='bar', ax=axes[1], rot=0, title='2. Trimmed edge segments (train only)')
len_res.pivot(index='length', columns='model', values='macro_f1').plot(
    marker='o', ax=axes[2], title='3. Window length (steps)')
axes[2].set_xticks([32, 64, 128])
axes[0].set_ylabel('macro-F1 (LORO 21-fold, pooled)')
for ax in axes[:2]:
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', fontsize=7)
plt.tight_layout()
plt.savefig('results/ablation.png', dpi=150)
plt.show()

In [ ]:
from google.colab import files
pd.concat([aug_res.assign(exp='augment'), edge_res.assign(exp='trim'), len_res.assign(exp='length')]
          ).to_csv('results/ablation.csv', index=False)
files.download('results/ablation.csv')
files.download('results/ablation.png')